# Module 3 · Fine-tuning, LoRA et QLoRA — Travaux pratiques
**Togo AI Lab · Formation IA · Version IT** · Durée : environ 1 h 30

**Ce que vous allez faire :** spécialiser un modèle pré-entraîné multilingue (`xlm-roberta-base`) pour classer des messages d'usagers (santé, état civil, éducation), en n'entraînant qu'un **petit adaptateur LoRA** (moins de 1 % des paramètres), puis comparer avec la base de référence TF-IDF du module 1.

**À la fin, vous saurez :**
1. préparer des données et un jeu de test honnête pour un fine-tuning ;
2. configurer LoRA (rang, modules ciblés) et mesurer la part de paramètres entraînés ;
3. entraîner, suivre les métriques et évaluer sur des messages **reformulés** ;
4. sauvegarder, recharger et fusionner un adaptateur ;
5. comprendre ce que change QLoRA (bonus avec GPU).

**Comment utiliser ce notebook (Google Colab)**
- Exécutez les cellules avec **Maj + Entrée**, dans l'ordre. Un **GPU** accélère l'entraînement : menu *Exécution* > *Modifier le type d'exécution* > *GPU T4*. Le TP fonctionne aussi sur CPU, plus lentement.
- Les cellules **👉 À vous** contiennent des `____` à remplacer. Toutes les données sont **fictives**.

## 0 · Préparation

In [ ]:
try:
    import peft, transformers
except ImportError:
    %pip install -q transformers peft accelerate

import os, random, re, unicodedata
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
APPAREIL = "GPU" if torch.cuda.is_available() else "CPU"
print("Outils prêts ✔ — calcul sur", APPAREIL)

## 1 · Les données
Pour ce TP, un petit **générateur** fabrique des messages fictifs à partir de modèles de phrases (avec fautes et accents oubliés), pour trois catégories. **Exécutez la cellule sans la lire en détail.**

Deux jeux de test :
- `test` : messages construits avec les **mêmes modèles de phrases** que l'entraînement (facile) ;
- `test_reformule` : 18 messages écrits à la main, avec **d'autres mots** (difficile, plus réaliste).

In [ ]:
# --- Générateur de messages fictifs (ne pas lire en détail) ---
def sans_accents(t):
    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")

def bruiter(t, rnd):
    r = rnd.random()
    if r < 0.15: t = sans_accents(t)
    elif r < 0.25: t = t.lower()
    elif r < 0.33: t = t.replace("vacciner", "vacsiner").replace("médecin", "medcin")
    return t

SLOTS = {
    "pers": ["Mon enfant", "Mon fils", "Ma fille", "Mon bébé", "Mon mari", "Ma mère", "Mon voisin"],
    "perl": ["mon bébé", "mon fils", "ma fille", "mon enfant", "mon petit dernier"],
    "symp": ["de la fièvre", "des maux de ventre", "une toux persistante", "des vomissements", "la diarrhée", "une forte migraine"],
    "duree": ["hier", "depuis deux jours", "depuis trois jours", "depuis une semaine", "depuis ce matin"],
    "lieu": ["un médecin", "un centre de santé", "une infirmière", "un hôpital"],
    "heure": ["la consultation au centre de santé", "l'hôpital régional", "la maternité", "le dispensaire"],
    "mal": ["le paludisme", "la toux", "la fièvre typhoïde", "la diarrhée"],
    "doc": ["un acte de naissance", "un extrait de mariage", "un certificat de nationalité", "une copie de mon acte"],
    "dem": ["déclarer une naissance", "célébrer un mariage", "renouveler ma carte d'identité", "obtenir un passeport"],
    "ville": ["Lomé", "Kara", "Sokodé", "Atakpamé", "ma commune"],
    "evt": ["la rentrée scolaire", "l'inscription à l'école primaire", "les examens du baccalauréat", "le concours d'entrée au collège"],
    "pap": ["un certificat de scolarité", "un relevé de notes", "une attestation d'inscription"],
    "aide": ["une bourse d'études", "une place à la cantine", "un transfert vers une autre école"],
    "niv": ["primaire", "maternelle", "secondaire"],
}
MODELES = {
    "sante": ["{pers} a {symp} {duree}, où trouver {lieu} ?", "Quels sont les horaires de {heure} cette semaine ?",
              "Où faire vacciner {perl} ?", "Je cherche des médicaments contre {mal}, que faire ?"],
    "etat_civil": ["Comment obtenir {doc} pour {perl} ?", "Quels papiers faut-il pour {dem} à la mairie ?",
                   "Où se trouve le bureau d'état civil de {ville} ?", "Je dois {dem} avant la fin du mois, est-ce possible ?"],
    "education": ["Quand commence {evt} cette année ?", "{pers} a besoin de {pap} pour son dossier.",
                  "Comment demander {aide} pour {perl} ?", "Où trouver une école {niv} près de {ville} ?"],
}
def generer(n_par_classe, seed):
    rnd = random.Random(seed); lignes = []
    for cat, modeles in MODELES.items():
        vus, essais = set(), 0
        while len(vus) < n_par_classe and essais < 5000:
            essais += 1
            m = rnd.choice(modeles)
            t = m.format(**{k: rnd.choice(v) for k, v in SLOTS.items()})
            t = bruiter(t, rnd)
            if t not in vus:
                vus.add(t); lignes.append((t, cat))
    rnd.shuffle(lignes)
    return pd.DataFrame(lignes, columns=["message", "categorie"])

TEST_REFORMULE = [
    ("Mon petit a le corps brûlant depuis hier soir.", "sante"),
    ("Ma grand-mère tousse beaucoup et respire mal.", "sante"),
    ("Où piquer mon nourrisson contre la rougeole ?", "sante"),
    ("Le dispensaire du quartier est-il ouvert dimanche ?", "sante"),
    ("J'ai besoin de quelqu'un pour soigner ma plaie infectée.", "sante"),
    ("Mon voisin vomit et se plaint de la tête.", "sante"),
    ("J'ai perdu mon papier d'identité, que dois-je faire ?", "etat_civil"),
    ("Nous voulons nous marier le mois prochain, quelles démarches ?", "etat_civil"),
    ("Mon fils vient de naître, comment l'enregistrer ?", "etat_civil"),
    ("Je dois faire établir un document pour voyager à l'étranger.", "etat_civil"),
    ("Où est le bureau qui délivre les extraits de naissance ?", "etat_civil"),
    ("Comment prouver sur papier que je suis le fils de mon père ?", "etat_civil"),
    ("Ma fille doit s'inscrire au lycée en septembre, comment faire ?", "education"),
    ("Quand tombent les résultats des épreuves du bac ?", "education"),
    ("Mon fils a besoin d'un papier de son école pour une bourse.", "education"),
    ("Y a-t-il une cantine pour les écoliers de mon quartier ?", "education"),
    ("L'instituteur de mon enfant est absent depuis un mois.", "education"),
    ("Je cherche une formation pour mon neveu qui a quitté l'école.", "education"),
]
LABELS = ["sante", "etat_civil", "education"]
label2id = {l: i for i, l in enumerate(LABELS)}; id2label = {i: l for l, i in label2id.items()}

train = generer(100, seed=1)
test = generer(20, seed=2)
test = test[~test["message"].isin(set(train["message"]))].reset_index(drop=True)   # aucun message commun
test_reformule = pd.DataFrame(TEST_REFORMULE, columns=["message", "categorie"])
print(len(train), "messages d'entraînement,", len(test), "de test,", len(test_reformule), "reformulés")
train.head(5)

## 2 · La base de référence : TF-IDF + régression logistique
Avant tout fine-tuning, **mesurez une base simple** (module 1). Si votre modèle sophistiqué ne la bat pas, il n'en vaut pas la peine.

In [ ]:
base = make_pipeline(TfidfVectorizer(lowercase=True), LogisticRegression(max_iter=1000))
base.fit(train["message"], train["categorie"])

def note(modele_predire, df):
    return accuracy_score(df["categorie"], modele_predire(df["message"].tolist()))

acc_base = {"test": note(base.predict, test), "reformulé": note(base.predict, test_reformule)}
print("TF-IDF — test :", round(acc_base["test"], 2), "| reformulé :", round(acc_base["reformulé"], 2))

## 3 · Préparer les données pour le modèle
Un Transformer attend des **tokens** (module 2) sous forme de tableaux de même longueur : on coupe (*troncature*) et on complète (*remplissage*).

**👉 À vous — Exercice 1.** Tokenisez les messages d'entraînement avec une longueur maximale de 48 tokens.  
*Indice : `tokenizer(liste, truncation=True, padding=True, max_length=48)`.*

In [ ]:
MODELE = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODELE)
print(tokenizer.tokenize("ou fer vacsiner mon bebe"))

In [ ]:
enc_train = tokenizer(train["message"].tolist(), truncation=True, padding=True, max_length=48)
print(len(enc_train["input_ids"]), "messages ;", len(enc_train["input_ids"][0]), "tokens chacun")

In [ ]:
class Jeu(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tokenizer(df["message"].tolist(), truncation=True, padding="max_length", max_length=48)
        self.y = [label2id[c] for c in df["categorie"]]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i])
        return item

ds_train, ds_test, ds_reform = Jeu(train), Jeu(test), Jeu(test_reformule)
print(len(ds_train), len(ds_test), len(ds_reform))

## 4 · Le modèle de base
On charge un encodeur pré-entraîné et on lui ajoute une petite *tête de classification* à 3 sorties.

**👉 À vous — Exercice 2.** Comptez les paramètres du modèle de base.  
*Indice : `sum(p.numel() for p in modele.parameters())`.*

In [ ]:
modele = AutoModelForSequenceClassification.from_pretrained(
    MODELE, num_labels=3, id2label=id2label, label2id=label2id)

In [ ]:
total = sum(p.numel() for p in modele.parameters())
print(f"{total/1e6:.0f} millions de paramètres")

## 5 · Brancher LoRA
Au lieu d'entraîner les ~278 millions de paramètres, on **gèle** le modèle et on ajoute de petites matrices dans les couches d'attention (`query` et `value`).

**👉 À vous — Exercice 3.** Configurez LoRA avec un rang de 16, `lora_alpha=32`, et ciblez `query` et `value`.  
*Indice : `LoraConfig(task_type="SEQ_CLS", r=..., lora_alpha=..., lora_dropout=0.1, target_modules=[...])`.*

In [ ]:
config = LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.1,
                    target_modules=["query", "value"])
modele_lora = get_peft_model(modele, config)
modele_lora.print_trainable_parameters()

**👉 À vous — Exercice 4.** Quelle part des paramètres est entraînée ? Calculez le pourcentage.  
*Indice : `modele_lora.get_nb_trainable_parameters()` renvoie `(entraînables, total)`. La tête de classification est entraînée en plus de LoRA.*

In [ ]:
entrainables, total = modele_lora.get_nb_trainable_parameters()
print(f"{100 * entrainables / total:.2f} % des paramètres sont entraînés")

## 6 · Entraîner
Le `Trainer` de Hugging Face gère la boucle d'entraînement. Avec LoRA, on utilise un taux d'apprentissage **plus élevé** qu'en fine-tuning complet.

**👉 À vous — Exercice 5.** Choisissez le nombre d'époques (essayez 10) et lancez l'entraînement.  
*Indice : une époque = un passage sur tous les exemples d'entraînement.*

In [ ]:
def metriques(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": float((np.argmax(logits, axis=-1) == labels).mean())}

In [ ]:
args = TrainingArguments(
    output_dir="sortie_lora", num_train_epochs=10, per_device_train_batch_size=16,
    learning_rate=5e-4, eval_strategy="epoch", save_strategy="no",
    logging_strategy="epoch", report_to="none", seed=SEED)

entraineur = Trainer(model=modele_lora, args=args, train_dataset=ds_train,
                     eval_dataset=ds_test, compute_metrics=metriques)
entraineur.train()

In [ ]:
historique = pd.DataFrame(entraineur.state.log_history)
cols = [c for c in ["epoch", "loss", "eval_loss", "eval_accuracy"] if c in historique.columns]
historique[cols].dropna(how="all", subset=[c for c in cols if c != "epoch"]).groupby("epoch").first().round(3)

> **Si la précision reste proche de 33 %** (le hasard pour 3 catégories), le modèle n'a pas encore appris : augmentez le nombre d'époques (par exemple 15) ou le taux d'apprentissage (`1e-3`), puis relancez l'entraînement.

> **Lire les courbes.** La perte d'entraînement (`loss`) doit baisser. Si `eval_loss` remonte alors que `loss` continue de baisser, c'est du **surapprentissage** : il faudrait s'arrêter plus tôt.

## 7 · Évaluer honnêtement
Comparons la base TF-IDF et le modèle LoRA sur les **deux** jeux de test.

**👉 À vous — Exercice 6.** Écrivez la fonction de prédiction du modèle LoRA.  
*Indice : `modele_lora(**entrees).logits` donne un score par catégorie ; `argmax` choisit la plus forte.*

In [ ]:
def predire_lora(messages):
    modele_lora.eval()
    entrees = tokenizer(messages, truncation=True, padding=True, max_length=48, return_tensors="pt")
    entrees = {k: v.to(modele_lora.device) for k, v in entrees.items()}
    with torch.no_grad():
        logits = modele_lora(**entrees).logits
    return [id2label[int(i)] for i in logits.argmax(dim=-1)]

acc_lora = {"test": note(predire_lora, test), "reformulé": note(predire_lora, test_reformule)}
pd.DataFrame({"TF-IDF": acc_base, "LoRA": acc_lora}).round(2)

In [ ]:
messages_r = test_reformule["message"].tolist()
details = pd.DataFrame({"message": messages_r, "attendu": test_reformule["categorie"],
                        "TF-IDF": base.predict(messages_r), "LoRA": predire_lora(messages_r)})
details[(details["TF-IDF"] != details["attendu"]) | (details["LoRA"] != details["attendu"])]

Observez les erreurs : quelles reformulations trompent TF-IDF ? Quelles reformulations trompent LoRA ? Que se passe-t-il quand on change **tous** les mots ?

> Les résultats dépendent du matériel et de la version des bibliothèques. Ce qui compte : **ne jamais conclure sur le seul jeu de test facile**.

## 8 · Savoir dire « je ne sais pas »
Comme au module 2, un score de confiance permet de refuser les messages hors sujet.

In [ ]:
def predire_avec_confiance(messages):
    modele_lora.eval()
    entrees = tokenizer(messages, truncation=True, padding=True, max_length=48, return_tensors="pt")
    entrees = {k: v.to(modele_lora.device) for k, v in entrees.items()}
    with torch.no_grad():
        probas = torch.softmax(modele_lora(**entrees).logits, dim=-1)
    conf, idx = probas.max(dim=-1)
    return [(id2label[int(i)], round(float(c), 2)) for i, c in zip(idx, conf)]

nouveaux = ["Je veux un prêt pour acheter une voiture.", "Où faire vacciner mon bébé ?", "Quand sortent les résultats du bac ?"]
for m, (cat, c) in zip(nouveaux, predire_avec_confiance(nouveaux)):
    print(f"{cat:12} confiance {c:.2f}  ← {m}")

## 9 · Sauvegarder, recharger, fusionner
L'**adaptateur** est un tout petit fichier : on le partage sans le modèle de base.

**👉 À vous — Exercice 7.** Sauvegardez l'adaptateur et mesurez sa taille.  
*Indice : `modele_lora.save_pretrained("dossier")`.*

In [ ]:
modele_lora.save_pretrained("adaptateur_lora")

def taille_dossier(d):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(d) for f in fs) / 1e6

print(f"Adaptateur : {taille_dossier('adaptateur_lora'):.1f} Mo")
print(f"Modèle de base (en 32 bits) : environ {total * 4 / 1e6:.0f} Mo")

In [ ]:
# Recharger : modèle de base neuf + adaptateur sauvegardé
base_neuf = AutoModelForSequenceClassification.from_pretrained(MODELE, num_labels=3, id2label=id2label, label2id=label2id)
recharge = PeftModel.from_pretrained(base_neuf, "adaptateur_lora")
recharge.eval()

# Fusion : un modèle classique, sans dépendance à peft
fusionne = recharge.merge_and_unload()
entrees = tokenizer(["Où faire vacciner mon bébé ?"], return_tensors="pt")
print("Prédiction après fusion :", id2label[int(fusionne(**entrees).logits.argmax(dim=-1))])

## 10 · Bonus : QLoRA (GPU requis)
QLoRA charge le modèle de base en **4 bits** pour économiser la mémoire, puis ajoute LoRA par-dessus. Cette cellule ne s'exécute que sur GPU ; elle compare l'empreinte mémoire d'un petit modèle de langage.

In [ ]:
if torch.cuda.is_available():
    %pip install -q bitsandbytes
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    NOM = "Qwen/Qwen2.5-0.5B"
    normal = AutoModelForCausalLM.from_pretrained(NOM, torch_dtype=torch.float16, device_map="auto")
    print(f"16 bits : {normal.get_memory_footprint() / 1e6:.0f} Mo")
    del normal; torch.cuda.empty_cache()
    q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    petit = AutoModelForCausalLM.from_pretrained(NOM, quantization_config=q4, device_map="auto")
    print(f" 4 bits : {petit.get_memory_footprint() / 1e6:.0f} Mo")
else:
    print("Pas de GPU détecté : cette cellule est facultative. Activez un GPU (menu Exécution) pour l'essayer.")

## Bilan
Discutez en binôme :
1. Sur quels messages le modèle LoRA a-t-il battu la base TF-IDF ? Sur lesquels a-t-il échoué ? Qu'en concluez-vous ?
2. Combien d'exemples avez-vous utilisés ? Que changerait-il d'en avoir dix fois plus, ou de meilleure qualité ?
3. Quels risques de confidentialité si les exemples étaient de vrais messages d'usagers ?
4. Quand préféreriez-vous un RAG (module 5) à un fine-tuning ?

**Prochain module :** NER clinique et OCR.